# Price Tiering for properties


**Dataset**: full_merged_V1.csv (https://drive.google.com/file/d/1Pp6nDx-vK-6rEFez_5gLpmtZr59BLhqA/view?usp=drive_link)

**Instruction**: Download the CSV file above and move it to the ds-chapa-affordable-housing/fa25-team-a/data folder, then run the cells below

This notebook performs the following tasks on the full merged CHAPA resale dataset:

1. Drop rows with missing `Town` or `Address`.
2. Clean and convert `Maximum Resale Price` to numeric.
3. Fix encoding issues in `income_bracket` and `asset_bracket`.
4. Identify and remove invalid or zero resale prices.
5. Create 4-tier price categories based on quartiles.
6. Save the cleaned dataset.
7. Optional: Show summary statistics for each price tier.


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("full_merged_V1.csv")
# this is the output dataset from Ria_Merge_PriceResaleApplicantData_OriginalWorkFlow.ipynb

# Drop rows with missing essential fields
df = df.dropna(subset=['Town', 'Address'])

# Keep original price as string
df['raw_price'] = df['Maximum Resale Price'].astype(str)

# Clean non-numeric characters
df['Numeric Price'] = (
    df['Maximum Resale Price']
    .astype(str)
    .str.replace('[^0-9.]', '', regex=True)
)

# Convert to numeric, coercing errors to NaN
df['Numeric Price'] = pd.to_numeric(df['Numeric Price'], errors='coerce')

# Drop rows where conversion failed (i.e., was a string)
df = df.dropna(subset=['Numeric Price'])

# Optional: reset index
df = df.reset_index(drop=True)

# -----------------------------
# Step 3: Fix text encoding issues in brackets
# -----------------------------
df['income_bracket'] = df['income_bracket'].str.replace('‚Äì', '–', regex=False)
df['asset_bracket'] = df['asset_bracket'].str.replace('‚Äì', '–', regex=False)

# -----------------------------
# Step 4: Identify invalid or zero prices
# -----------------------------
invalid_rows = df[df['Numeric Price'].isna() | (df['Numeric Price'] == 0)]

if not invalid_rows.empty:
    print("⚠️ Rows with invalid or zero resale prices:")
    print(invalid_rows[['Town', 'Address', 'raw_price']])
else:
    print("✅ No invalid or zero resale prices found.")

# Drop invalid or zero prices
df = df.dropna(subset=['Numeric Price'])
df = df[df['Numeric Price'] != 0]

# -----------------------------
# Step 5: Create 4-tier price categories
# -----------------------------
df['Price Tier'] = pd.qcut(
    df['Numeric Price'],
    q=4,
    labels=['Low', 'Lower-Mid', 'Upper-Mid', 'High']
)

# Move 'Price Tier' column next to 'Maximum Resale Price'
cols = list(df.columns)
price_idx = cols.index('Maximum Resale Price')
cols.insert(price_idx + 1, cols.pop(cols.index('Price Tier')))
df = df[cols]

# -----------------------------
# Step 6: Save cleaned dataset
# -----------------------------
output_path = "full_merge_price_tiers_baseQ3.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Cleaned dataset saved successfully as: {output_path}")

# -----------------------------
# Step 7: Optional — Summary of tiers
# -----------------------------
print("\nPrice range summary per tier:")
print(df.groupby('Price Tier')['Numeric Price'].agg(['min', 'max', 'mean']).round(2))


✅ No invalid or zero resale prices found.

✅ Cleaned dataset saved successfully as: full_merge_price_tiers_baseQ3.csv

Price range summary per tier:
                 min       max       mean
Price Tier                               
Low         148438.0  201200.0  188379.20
Lower-Mid   201300.0  218988.0  207157.02
Upper-Mid   221600.0  266536.0  239256.85
High        268884.0  374062.0  302801.65


/tmp/ipython-input-1721232134.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('Price Tier')['Numeric Price'].agg(['min', 'max', 'mean']).round(2))
